In [0]:
from pyspark.sql import Window
from pyspark.sql import functions as F

# Leemos los países únicos que existen en nuestra capa Silver (SUNAT y Comtrade)
df_paises_sunat = spark.table("silver.sunat_paises").select("pais_destino").distinct()
df_paises_comtrade = spark.table("silver.comtrade").select("pais_destino").distinct()

# Unimos ambos y quitamos duplicados globales
df_paises_unicos = df_paises_sunat.union(df_paises_comtrade).distinct()

# Generamos la clave sustituta (id_pais)
ventana = Window.orderBy("pais_destino")
dim_pais = df_paises_unicos.withColumn("id_pais", F.row_number().over(ventana))

# Ordenamos columnas: id primero
dim_pais = dim_pais.select("id_pais", "pais_destino")

# Guardamos en Gold
dim_pais.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable("gold.dim_pais")

display(dim_pais.limit(5))


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


id_pais,pais_destino
1,AFGANISTAN
2,AGUAS INTERNACIONALES
3,ALBANIA
4,ALEMANIA
5,ANDORRA
